In [8]:
import torch
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt
%matplotlib inline

In [9]:
words = open('names.txt').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

vocab_size = len(itos)
print(vocab_size)

27


In [10]:
block_size = 8

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [11]:
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [12]:
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [13]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn(fan_in, fan_out) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        dim = (0,1) if x.ndim == 3 else 0
        xmean = x.mean(dim, keepdim=True)
        xvar = x.var(dim, keepdim=True)

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []


class Embedding:
    def __init__(self, vocab_size, dim):
        self.weight = torch.randn(vocab_size, dim)

    def __call__(self, ix):
        self.out = self.weight[ix]
        return self.out

    def parameters(self):
        return [self.weight]


class FlattenConsecutive:
    def __init__(self, n):
        self.n = n

    def __call__(self, x):
        B, T, C = x.shape
        x = x.view(B, T//self.n, C*self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)
        self.out = x
        return self.out

    def parameters(self):
        return []


class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [14]:
n_embd = 24
n_hidden = 128

model = Sequential([
    Embedding(vocab_size, n_embd),

    FlattenConsecutive(2),
    Linear(n_embd * 2, n_hidden, bias=False),
    BatchNorm1d(n_hidden),
    Tanh(),

    FlattenConsecutive(2),
    Linear(n_hidden * 2, n_hidden, bias=False),
    BatchNorm1d(n_hidden),
    Tanh(),

    FlattenConsecutive(2),
    Linear(n_hidden * 2, n_hidden, bias=False),
    BatchNorm1d(n_hidden),
    Tanh(),

    Linear(n_hidden, vocab_size),
])

In [15]:
parameters = model.parameters()
for p in parameters:
    p.requires_grad = True

print(sum(p.nelement() for p in parameters))

76579


In [16]:
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 150000 else 0.01

    for p in parameters:
        p.data += -lr * p.grad

    if i % 10000 == 0:
        print(i, loss.item())

    lossi.append(loss.log10().item())

0 3.6720798015594482
10000 2.686992645263672
20000 1.8993185758590698
30000 2.1252965927124023
40000 2.038532018661499
50000 1.8791393041610718
60000 1.623574137687683
70000 1.917995572090149
80000 2.1307175159454346
90000 1.7793489694595337
100000 1.5369195938110352
110000 1.747970700263977
120000 1.7673850059509277
130000 1.7525006532669067
140000 1.8302690982818604
150000 1.9539759159088135
160000 1.8918510675430298
170000 2.324397087097168
180000 2.0957159996032715
190000 1.9938703775405884


In [20]:
for _ in range(20):

    out = []
    context = [0] * block_size

    while True:
        x = torch.tensor([context], dtype=torch.long)
        logits = model(x)

        # 안정화
        logits = logits - logits.max(dim=1, keepdim=True).values

        probs = F.softmax(logits, dim=1)

        # NaN 체크 (디버깅용)
        if torch.isnan(probs).any():
            print("NaN detected!")
            break

        ix = torch.multinomial(probs, num_samples=1).item()

        context = context[1:] + [ix]
        out.append(ix)

        if ix == 0:
            break

    print(''.join(itos[i] for i in out))

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!

NaN detected!



/var/folders/lg/6_ftpksn6lgg9jbrnbmyz1hr0000gn/T/ipykernel_14362/602137725.py:25: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  xvar = x.var(dim, keepdim=True)
